# MVA v4 · NB02B — BUBR1 allele-specific skeletal-muscle consequence atlas
**v1.0.3 · public data only · targeted completion of NB02**

## Why this notebook exists

The original NB02 was deliberately broad. It compared general CIN/aneuploidy models and found **no universal
mitochondrial signature**. Its role is therefore retained as a *Downstream Consequence Atlas*, not as evidence
that mitochondrial therapy is causal in BUB1B-MVA.

This targeted completion asks a narrower question:

> In a BUBR1-specific skeletal-muscle model, do the same frozen NB02 modules show reproducible
> allele-dependent downstream states?

Public datasets:
- **GSE134781**: 3-month gastrocnemius, WT (n=3), BubR1+/L1002P (n=3), BubR1+/X753 (n=3).
- **GSE134780**: 8–10-month skeletal muscle, WT (n=4), BubR1H/L1002P (n=4), BubR1H/H (n=3).

Critical mapping note:
- mouse **BubR1 L1002P models human BUBR1 L1012P**, not N1002K.
- It is therefore a **nearby pseudokinase-domain / destabilized-allele comparator**, approximately 10 residues
  from N1002K, not an exact-variant surrogate.

## Frozen analysis

Primary gene sets are the **same 14 human GO-derived NB02 modules**, recovered byte-for-byte from the user's
original NB02 frozen reference. They are mapped to mouse through one-to-one Ensembl orthology.

Pre-specified contrasts:
1. 3mo L1002P vs WT
2. 3mo X753 vs WT
3. 3mo X753 vs L1002P
4. 10mo H/L1002P vs WT
5. 10mo H/H vs WT
6. 10mo H/L1002P vs H/H

Module effect = mean sample-level module score(group A) - mean(group B), where the module score is the mean
`log2(CPM + 0.5)` across mapped genes.

A module is called `ROBUST_DIRECTIONAL` within a contrast only when:
- |effect| >= 0.15 log2 units; and
- every leave-one-sample-from-each-group re-estimate has the same sign.

Exact label-permutation p-values are descriptive because sample sizes are small.

### Mitochondrial recurrence gate

`Mito_translation` and `OXPHOS` are called `RECURRENT_BUBR1_DOWNSTREAM` only if the same direction is robust
in at least two pre-specified BUBR1-vs-WT contrasts spanning both datasets/ages.

Otherwise the final interpretation is `CONTEXT_SPECIFIC`, which strengthens rather than reverses the original
NB02 conclusion.

No drug is selected by this notebook.## v1.0.1 technical amendment

v1.0.0 attempted BioMart through HTTP POST. The current Ensembl martservice returned HTTP 405
(Method Not Allowed). v1.0.1 changes only the transport layer:

1. reuse a cached orthology table if already present;
2. otherwise query Ensembl BioMart through HTTP GET (`params={"query": ...}`), preferring the Ensembl-116 archive;
3. if both BioMart endpoints are unavailable, use a conservative direct-symbol intersection only as an explicitly
   labelled **sensitivity fallback**.

No dataset, module, contrast, threshold, recurrence gate, or biological interpretation rule is changed.## v1.0.3 MGI duplicate-safe orthology patch

Both Ensembl BioMart endpoints were unavailable in the Colab runtime, so v1.0.1 correctly downgraded itself to
`DIRECT_SYMBOL_SENSITIVITY_ONLY`.

v1.0.3 uses the Mouse Genome Informatics (MGI) report
`HOM_ProteinCoding.rpt`, explicitly defined by MGI as **mouse protein-coding genes having one-to-one orthology
with human genes**.

This is a technical resource substitution only. The following remain unchanged:
- the two GEO datasets;
- all 14 frozen NB02 modules;
- all six contrasts;
- the `|effect| >= 0.15` threshold;
- the leave-one-sample-from-each-group sign-stability rule;
- the mitochondrial recurrence rule;
- the interpretation of mouse L1002P as a model of human L1012P, not N1002K.

The earlier direct-symbol result is retained only as a sensitivity analysis.

### v1.0.3 source-normalization safeguard

MGI's report is one-to-one at the gene-relationship level, but a source gene can occasionally occupy more than one raw row when an external identifier has multiple records. v1.0.3 therefore validates one-to-one identity using MGI↔HGNC identifiers first, then collapses only rows that are identical in the gene relationship and differ solely in the mouse NCBI Gene ID. Any true MGI↔HGNC conflict fails hard.

The previous draft also contained a non-preregistered hard `coverage >= 0.65` assertion. v1.0.3 converts that value to a descriptive QC notice only; the frozen biological decision rules are unchanged.


In [ ]:
import os
import io, os, re, json, gzip, base64, itertools, math, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
    PUBLIC=ROOT/"03_PUBLIC_ARTIFACTS"
    PUBRES=ROOT/"05_PUBLIC_RESOURCES"
except Exception:
    ROOT=Path(".")
    PUBLIC=Path(".")
    PUBRES=Path("./public_resources")
PUBLIC.mkdir(parents=True,exist_ok=True); PUBRES.mkdir(parents=True,exist_ok=True)

MODULES=json.loads(gzip.decompress(base64.b64decode("H4sIADgZs2oC/9V9W6/sNprdf+nnHGCLkuqSN+pacul2JNXeVTtoGE6PZ3IAt92xPUE6Qf57yLW+j5LqbLtnJoPG5KXWKkmk7iL5XRb/9x/sP//601/+23f/9Nc//Of/8gebNeYP/8lBG3nIsyL1WNUm89hlk8WK/m6PHqeqGVFgutR2NAfQ9oSNpy6PgbMlYOlSRy+KkRAjGAsmgodWtjBWUKqQDRNZnMjiJBcsiCnhSDjL1mdu3eF3jGLL3Y+G5Db6nWYWO8lyOxNbvzIr8z4SxP++Gf3qPDr/9PM/4jxyz45gts2HfBDa+XJ5PtbA4uTPMb90Y2SVZEKMLjG6JBZMdE2SBYKjBc2FpIIHQX/6eTMXOJC2zKMDqml7VNsuKNfe/O9U8vCW+eqgsKP/Lab66pcW5YidFb39xlUC1vlKi8ni7DzyP/ZVNiYBVGb2q8vWjsMk7BW3thxxj8qxw0mXrxaXo8rubUIc/NFXvTvN1pPhPvhy1XIhYOGtL/JIid9zbS1+MzvhFIRhl8qxXYanuR6nKD4G4q9t7U6lwLFcLE76UtgcD64n/twv9ex/uzrD6nk8v1gr1J4cNlWDi9S006332F8yX2EzVVg91R3hs4elzvwZX9tL629Lazu8V21t29lv3jb3SMCfczuhrnYaxykHma7Y3r1suBedO+w2j+1Ksw01K8+FzkDW0lU9YS78vruB16Bbugnvpif+KLvXLMKN6zPc1z4frF/fX9oJd8S9HnPPwkLNSnFwyv3R9QOem35k2Smr/N+hXvzvuPiLOLoj9HdqLPLi0Ph7O5Z3Q/BP0ljbDnjBOY6XyvgDHZtrnAeSCZkSkB5P99iMyRUnQ4aNvnn178HY4nka2/J6wcUdpyvvNYghweHN7qlBrcvYG79islmkaAUzIt4Oj7LcyPI4VnK2SnRJzSvo6FHWnYBZlOOiTeUi7zCZCcyf/VTP0dlj0w3cvCP0VfQSf9JPjf97OAg5pSQmklUJSiz4eky3LO8VW5Dq4U9rjl5e8BaA+H3OvJtzbifcmrnMDc7YkQMWXOK6xcs0N9OCJV0+oY7uNlVY4R+gs5DkCIKXae7vuOwO4xci1rrHZSEuNjopi2OwOkKdn+cFN3XGizQvthPAISxZgZVL09ckVxZbbjzW5R4diTiA27iYl4tnjwUbLvYe8SO9ZHnEp4PsqMykykjwPC65+zSAlPk4BWJA7uaAeqrSX8Glqe0E7Do87kt3nnHBli5rugNI2UXpSybUvJyE8RnzLIl0bXoWcsROhrbFm7YMuE7LmMYCTT9GK8WhuU/rmKOYe8h4io7gvGZp8W+ZTQn4WN+yz20fKQlLEpB774/+VuExvVUtAcdxa68ChuCP6jaP+Ep7lP94IG6vk60dvlq2hx79JXjN/Zm9djiP13GOFPFxAsmVFEIOgici7p5H+R9r0Vg2jI+6AE0LSK5EKo3PxET2n0iRREqkWL5YQoNPiUe/+q2oHrFgApyMIKrzmCqywHTCimZsFP01vPM5fQx45N+rx2sZCLp271f3+uLFf/ffgdMf/ojewLf/8N2fv/un77/9+ftf/vLTj798z44k+26uw3i3MzuT6JXlJZ5hmy8t+iGOTKmgv4K2GCffFtnSf1hsXfon2zZ4G+11wZNo22uGDgCIURIrOSpBhf21LdlpHWN2JMeURzSm78QsI6L/6JqSSBBVj20FWO4eZn5oPRpB7Ne9cKh7vpgWuHCniy1iK4T90cps+p9d00vvk33aib88zWXCLpc7rrm99VicuR5MFylBN9TmF+m0HgBX/psK4lIB7vh9x0OTZTn7tTnrlP6tAzxjnrQreQksLDOBfBptNvayQjrLB4LFVyhrCvxOuGIesVXz2n0qpzz3VyVr/eXI3JPHvQ6+3XXPzoJ/U24VDZE1TEVCOAMa9tynriTOvAwLWsE8ilzHPEW3O0ocxSuXR5+zER30eTSCWJ61+MWT4CAmoGvtWjBsmrteJ5bnfcHOfX8hXNHFd7vJSPD6OUxb4pFguZiHnRdXAUOICan7ogo9EFj6eib0HEB4kgkxFvcU1F/5vOzHWfAOHCO0H44cUHe54MObV65Zxc3Oa4sSF1vJAMURDjryS8GRRYGeqEMOZUoev0NUdZl69AXzBu9W3hRoq3LXUABkmNMOOa5WO4/+RciHruPFnGyBY5e7mk8PwOx6a6UQPqpkJjCc0NJy58uINz2/tRw7OcRRPCZ8Xoqs8psVeTuVOE+yLLAcbMTbVeS3nk10UeCcHGC4UzRLLIjainskwGETPz9Fzd5MUV/fPXTsFxfsgrnPKOrqF99hKYZ+HnCAw4KrXCz8vcfAR9/iPSke05VH+2DPvTR44xz4p6QsR1zMsuZgr2yKWEZk9oqxWTtwe74yDlBJNz889MVQC75ilBa/vAATwZxjOLzW5XjBGfg3OhLU/7FgIpgKHhTbQLSIv1Llwl52eed4kmM/Bylh5oisfHCrB/f/wBPgACNI664yjtYzEwjXdVHE8SZYBoYPmYPcCsriPBfEsXhSClaCtWAj2Ap2GMlywOoGtLESfBU8UTwQ3zDkLXkg/otSNXWP210104Ta2scbv/hVhxe36tBDrHpaY9wwGQ2EQ9z3MGwe7iPHzYP/fZ84IC6KJMWAOe8B3HVdoXNQ1/45rJt+NoL+2tWdP7u6zyIih9bs2deuhcPBgYQlJl/Zt2FprCRRkoLMeCAvxvURU5L78yg8BtYVnplL2WaEdiZ+BrDNuTR4Ji9NX43A8RoJYnWLFvLS5aUM6jmSd2i2g3yH+D/ipblMA3bJrrYDg6bksthFFrneMLa/4SPkAQVubzi5psQguqkq3JxgL6gvXYbb2FyzaylYw4KAlqxhN6Dph9OLYqYkV1IoQQ39AjOWxyNtEDHB//um86/71S4pwS+7Fh1eDYdGEFa1q/t2GKB/LK7dgh6lR7/nq7z8bVMLxAR/d9sOj1f76GFqyCf2M8mMsgu/6p0t2Mdwn0mDls+R+Io3GOxIco2MkkRJKiQWtLzLji6AGR/TLkPD1WULDyOTtsJ9jydAy38dHjiHhhATEkJKOBCOhBPhTKBNyo1lpfqFSJtVV7DeghWWDZ4pjzDM8FXsymVpeX6e4QhqfrId4qQafK+6ZorOsbAYTV7X4l10gGLtvODoOloCHBr0V929QV+r6wu7NfkMfIEcVkkbBYZlY43f3j8XnWsssXri09nNePMdxISEkAp8ml1vmxd+vuDqzayUY5xuGXATlqn0731348W6zSfi8vAfxu6Bi/TAl7O3HL54hBkp68UkhbUFR9N9WeC292XTRoJGMAZyfO+Qm90mmCL7iq+HR1yRvpquMFpdym/EvoU+Y3+9x5+wgDfIAeod+gHA0XM/2pm2rk5sXaMiFk8FWuKeVtt+dl+mSElYEivBje6XC8/oxu5/f8uvM0mxEGk+GGRcMNS4IoNYMhziyoy2RrswsprRtjSl2aneGuDsNEaKL0JkO/RXPSaCZ0F8H0d7b6To3d/b0Q8TgPg8jMWMT8pYtmh/xkulgGovbWFpyeOw0iPteEXB/9ye1oGR1oGR1gEHMA52Vo4ArdjYsaIOd2TsrzjKoX4HtDT0DWheHBTyFzfGYyyYCLIJdqwU0L+yYU24EBrCldASOkJP+EyYDLYdxwivuidTlAo1E76/jiWKU2wDywKTtdx64ks+TgW+VePUdG5HNGditO6w4JbdEgkegK4JRJGZX6hx7qyiEYwFE8FU8CB4JGZSLpNymZTLpFwm5TIplx13hta5K2lwXfA0j29vNCyPDzdOK2AjxS3zcBQ8EU2k2CqLrRJaXIv0RTBS5DUD1W0wTACRHaZGMNFNkpZETLqFWHRn9hSmLMPL4jEVPAjiYLOO1uKsi/kfQ4wpvzxo60V/2cHnVjFRgvrKV273GnMD1/v0SLuaA0OICbIOJas3POHTpUcvZeLrNbmeBJE9qalrBFBTb+fyQrs2abbSXAzLkVqeaWvzjOfmrc+BnEnwf8Qj5sAQYgKOdmxhoZrG2RxbEq6eD1fLNWxSXA86IhgBHNtS8hnw+GlxH/e5OmDF7ZWWLBCYhG13oU2Y3+i5pPm4nHv+XfhL95Ensrg4EjsYax3xX7+56vE7fgbgyztf+JXxaARhsr6M00Xs42onTwQPgthHm8didG9pN29pe2+rnsbr9s6Dc5gRUY87MholyHgQoC1pzmJ07DrgthjHOZDL0tFQ3t2I/RtAhmEwwKMHOY/FA7W5L/dEdANN2ODHgYc5TnD6zDPbmtX2fqUJ3+Fpb4vnazTfOjys3hofHXDJbjOsFvPtNT6jQzS/NTj6t5kv9PzIacymzcoBrOAWxpzFDjTKOzSCMIIXNX5ZsqAJvOwXNMhLOVVGkRb08o7O6caAb0digUZzqVzDhLoq9FWXJp+mvYW/bMt5Bh3RBXc4+Qd+YYO3sMGDvf8FZ4Bnmbb93nUIxBchlvuhn1F4oDnGYayYEWX7YRxglt1Y/2UNTUjLeIxp/4fZxmEl/oAqFTwQeSUmGlk9cruGt8F7C2BJ9wQmOm8MjYwS7oRu3WXBA7jQonqzaBdv9Ok6OBCOgNJYwUwQr5QnpRHSCy6CN8HXSIlu+bb3UsCAfcsv+Naql+JCn9WtoYHn1vtH5cah423C59J7KZ6cFgciLoF3XhASxUQwFTwSU6lG/p7vG5fH7XVGAMarfY1Xj0c+sgv0Ol0FDAGeBJvDTcAX+q2Y4lj8BwIn4tEf7dsFH603ulEdGAKKsJV4c/cLj83b1POXO79bNPr3Brfz7t34fSCf3vvq+JKW4yfb1lF7xiajxa8/vLuMNe9iaLqLoekuhqa7GJruYmh68AF7oMV8z5ZMEd8QTxLFFyFw3Lzn8YVe7vf8bcTH7L24XBgBAYZB6cZLUvPOv3cWdsH37tHDj+5OKD6hx+bdJ7EhOaCxe59sj57Pu/swuXGkeFZ+/v4vP3z503e/fvnpR3hVisnShVHVAbOtn8S9NVtHyd/yVgSnxLT6G1a/wTs2erbNP1njc9vNYqK2bFvzIo+TveX78JGF21i1fNOWjWflK2P1ByZoNTVPNidZKvTpPDltjMxPVl+19qr1Vuyzl2oC3M9bO6yPcTnuTLFyLQvaC8Qie1u+Np9+aLbc2B8/NDvuzIRigRNo+jl6ssHFO1tcD9Pa9IYDU0sYbVpqzrm4e9IGu82TfUZKrjaX/3ejB4Z6wUgRrAxqJRDrgIznfYgIx652Htehe9VYQkbICXeMo3k6/ZJ2u7Huo59whsNQ+poGxqGIFWOg58lBQkgJhw8HtE/j0gvDTXR4GMaF1vz7jg/LZDdOrM3fHi8+DRRhEfKDuUjQ7Ad3S+dfsPE22f146OuxTv40NtExyXE/NpmjD8ck/8LByL9iEPLBKCN6Gm/83iBh7fr/bp//6x7+suu5XzgY/pd1p9t9T5adprVHOzFKJnRUl0yiPkLP7+se3/Wp6yYdtAj3fe2yPXe13tiVfeDuf9Qj6n+jv3PvT5uuy9pZSdb+xKavEJ22fYanBl/a6ND8MrJFGj/X5jU//vjdr99/++XPf/7nH7/8KiGpOW3+NqfB2Bbuu5kKWWZGjRa4NR7oyi9qJbzpluPWEEjApscjYgSuruHsGUjAAa1Fd831/nlejky0H/jQVwkWQJjCyJA3h1nJeDuh2UrzlRYrrVZar/RCyuiJz3jzLB8JBxpqWzKGYGqK1G5Dbuf5KeZgjHgV7ohJeG/Ea0+H/tjwt2J77kgWSCzEH3rmHt8HEGFZ2XKl59xuHejRZ8SRwnbufuEaRlAhXmvc9ZytMT2wSTbKv1H+wymE953dBfi5Ed6W4+zyU434WvQ08gXvwkcxtrZjHJ8jdO1P6H15PIuLPxJMxI9/pkO/FaBXv2XfJWJXJjq8EI/0v0cn+Q8vNYNrC/PyIsgejTHs+JiElSRcnXBpwppS7ifFQdK3XhyTTZRwmcHSnle4ChV2CMeg+8XQI68a/I5bz71JscmlZuepoVv9csPNa5oFi5ul5PK26BVDnPCLFWaUJIHkSgolpZJOSKobH5QcSXjDGXHcj4hdygfsMLAoMBzNMA3sA06MBcDCCV8eHyAAd6YjHC/lE5bTsuYBj+Q9lsJ3efxBDkKAD8TUOmCIQMbO5aPFKT5Gc2S/0Q1a8GEtLD8VhbiaCphMi3xeol0UgBGM70Q8SEXJtw6YCYkFE8FU8EDMIsUXvN5Cs0CTdWmyLk3Xpem69LAuPaxLj+vS47r0tKG50uglMBNYHFgSWBpYOA+0EWRnZSacnQm1mFCLCbWYY2ChljiyKw0HG4fjikM9cagn1qMJV43X7EIbjccDMZb/sfzHm6rjhS5bNCie909jNG4DH4AbOu1lPjcYKvA9YwiGkegKBmmMdBBWuJyVmejypz2lsnPL+AL/1lQ5WrsqL6cIS/N64ojCkZiE4QNMU3CAMAB8CqoaI44aNbD9regYqsWk45Fh9GNxAbrGkRECPnoWgfPcULbLRlmYEFLCgYDIeoYr1N8w4P6bnHH3KD2W0zZ6YC66Aoi4g/cuI1wJHYIBMokC8CRSYgJplcVKEiWpkqOSkxCjFeGb7YmW1sJaVtdrFVrDGekCr/w8XRDXdHFfAwlXSCQeAbWUd/bOL4xDaO0nK5gJ5oIFO7Selh/FIjjEcTIg+/LKGx+CCaoG74tDPMUOZfFFNlsCZkKMYCwo5RglDaJbdNykt7gbIImSg5KjEKMYCdFNU0EtoQVOgpMWmEwgnxr3EZ94xH0mG5SEWkDL1VruSmhleSuLWzmLVg6o1YKw9fhwDInByD6KyQjH0UZGkV1Px/A59VgJTrKABkxPeBQtPU8eJyKuQotOtIOEcLCCmWAtiG72mmliX5lpQheux4ThHtEu+CNdY0CeslLmGm+KQ9gsmiW/MEsF0fHfsN5v+AVz4Ov7Jr9YBoVIIElTMTakmeIC1/wqWSEejWAsyOQXGp4cVoqyXS3/sd8r3R8e/DNzffQ3pMxI7An6oy2/fMifiZWcQWDFbN3FwiMJEpakzKXhd4IWlPZxKAGnA+DMX/7Bd5gRLe1jZnhJu0QapCKRKInEn7ixcaVsYCgKoywk+rSzC/Nq7CvScTKxxgyMi+/K6nUXA1Jd6AMP0R8Mo2kqCQVJDhIKckST5UhbLmjXO2Z6dGIR6UYmjTCMrJttvxTxp8XtsOICPtEduzYeEJ2BBq6DAb97FCe/j14y0jwiPsHWVwC6TH1eMU8Il6HPGahQ+s680WgLlKqu2KHH5h1RFcji6+VJ8ZgSubdWjDaeGMVMSCx4EDwT7wzPyPg7ARqmIUWbbCQN3GC8xueBcRoRAlp6CZ3gbRgY0zEwZdBBuyYuaXSFw5YGqruGWRRN8mx/2gVOZOiAjjkb27EoIyPxEbAY1e1DSoKZwNa1CTOiok1iVPNVNhQd/p7Umh/FeAqakb6hnau1pkZLAWZgIGtzGqnE1iWmLmRLcfmISy/RFkMrkQmeZEpyJYWSUkmlpFZyUQIzmEamICAikVCHAyr8nIUAB0nXKqJdWMMVH+R/j+gF2gz/9WEM2UmQ4RR0FnmUenKpJ5d6cqknP2gYhGIgRkn8FClR6Bpdocu1Lg2tkGMqpUqaKOdRjm0R39G4IC5+fJ228Rbjg5FtHs0us61cELoNUithXASDCuw0lTPLzPUkVscuovWQxsTWElBjxZROj0YwFkSnniRsw2CMpm4A9OBOEgIKE2IgsZKDkqOSk5JWyWI0fCH9reQ5ODKngV7pyR01Qx7c4Z41SCHjKrjqptnS8e+T5yLzcT6d7Yrzzlgp3pfZDZoZqucz6yIbbWMU+pFBCK5DTx8+WLoNVNBQhGUsYOw0EsjgCaybl6oVKyi2b2odus9Nj/Oam2m02zy+68jwA9tJOIIjByUSsRDJkTqWWhNYHFhCZgwjFBw72EOIdjiRuRaPZWHb2gY0NB0zA7tRohTGosXnZ2aot08d3MUpNFNJ0+3AuI6Jo32P+D/nacFMQVqDHZp9piCzynzoAY79ccVvnzO7J4QW8LYtbC626YJXAfmvyX45s39A4BdvKvjsG/GKs//q4YVoCDEhIaSEA+FIOBEk/69gCqF0AhhScNB0wci8KI1jTSEMJGW8QcVf1/if8IZKMMI+rVAiCtxHICCz6j1NdsEDxWskaAS5HR9vj20gLFnLlrVsWb/q8lTwJCixBq3mLr4o0bAEiT1o+EECSZ8THU2IYTBKtJTRjc1ByfE54MGcNfJB64m1nlh3Gms98XOYRKebJHrsiVaTxIFkysLWWk8S6jmHjc55YEW0UrNN7PSoe0yjQGplYWPdYxqK6aVI9RhSPQZdc9CaD1rzQes76EkdkkCywPTAD7qzg+7joPvQ63/UGo9a41FrPGrpo26stbR6PHyyF2a9Lm3LYJUbrIgeeEwPxuU8pgHjkxsdECE+pRCXS3t4csYw1KTFVi2/BB6NIAJDugHF+/wcZ88BJRIxEklAiZHl5ngXct6Hlhx3WbJ431572BReZ8bxv1WF2IbJMmWREqMkFqIL9H8qeGaMCGxLOyfRLnBjuRQYL7zz9IPzKI8vPMD3PL/Qr4TAjEA2LAv0pCyV+IuTJLQezkcSNzD4oxvP/fWXn3756c+SzZpb+pkyBj141P8pER07j2eiOK0yRrl7hC8nY/qczW3PdFj8tg8AtVRy3FUHCUHTY8ULtjCBFnH67lf+dEyepRHNwtJna5gF7KW3Vzi5pojdVMsgHcsgHQfVIzi5DJ1cI9zncHap4IsRxOGrY2mMasFOkUmzkaTYRrP+x75GI94zUVtxozT5L+WNlItleVwIyvpUlqf6/ymVd6D2DL19NHE4OGzdaF6qJknFb7bzn03iNJym4iPtmmkuCDUBV3W2l+hJr2bkS+FJpH64KCzJn4VnDuKMBNNlry82WulmaRJoFlgeWLEWKtZC5bq0DEv10BzL1oV5tNJ1aRHYWlVUBVavCy9kJzn6yTbFNof5/QYXo0FIUVKLsSSzzG8IWcl4OYKjEiloWTvkfKjI4IocpnxOlRyUHAP5ZGd4xfHvtDo9J4x/xaFJ52RMQ2N+gJLPy1dSPnRp2hbeJIlcssx79miYNQzbS56z3hy5Nnn+Hglk4koUl6L8zbeORGa3FhJAxfTc4nDaegUv0TZvuHSfErSWZNiyQjzX6vSjlffvKDd0ZVxr3uaiMpT3+j8VPAgev3I3PgkUSQlmVbf0W4pqUbfwzIf2wFxgeP7GvowEUcM4ipMTI5sgcjRd6LmeHjajpzAWwMWbR7R2HnEAMwKp8iWDe3XpCfg+OMgIOaEgVISacFmVlRy0hIEglb0S3gjviF1DLFR+F6enw4TeydSN7WKh6NsXjKYsbN0y2q0v8AkoINFSFAPsk/8Cx+PIBtoRhswzGC5WFkhKwnEzCSoc6DAuBj5txYhnwcHxI7GouY/2qc15hAe85CehLCuODEHgMasrakjB4L9RlZJGy+HI/24Ii5TjsZi2alOqMmXpqaoYqhxcbHkH11hlcaWrFpevaoeFCbfoa0GRSjJud5m34lCbrl+rVDFs0iO2fORD9KFWVZsT8ISpRBWWYaBaF2wk6gLRHHXFYjUdczUcdk1n6YSDuURkdWo33qKnrUVgRM3MzrpnE++wUYyJnyXHl2AICRN+HwAc4oB43HoUpx4luEY8JzXFYBykBPgER0bJfaS7BZYLS2IhGDDUPkSH4lxwDd7w0tUPvM31e2d3/sLLxl8oEl40z4pjrmQf+lLeLSGjqFdvF/W8dTYw9coVQ1g2ZF976IoxC/Sz3VAT6GaDLCydQq3Tun6iY480WWkqFCnLzFl1kBAOBLgKp376WJrMISy2lweNYx71fywIh1Ex4/dmxYX48rddgZO4vuihSvBINd3IRqvp54kOKWYy7zXQ6F1oloorXeXIKc5pNbrm/Kx7NETkBTv8jOUlruK1yWezdT+14mu6tExcpj3+escj6HXWoq3emmXz5zElLvy6kJnA4sCSwFhiXDq0nGSZMq6kj78t0I1o647RuDCXtc2CgFNggYTpjA6xVr5s7ZjT2LTKv/Efj32aGC4FUm5V4W7ds5uqffjWpTuMcOTYWnxWPbvjIPRI9Zn9Sk7uGoXM6ig4tqKvXFyXKg7sFNhZXVqRknibHl1apnZXM2OSPaoYHb5XQZWuFm9UI57hrhngNWvgtutapgyHhOdim91Mf5p6vcaBscnIZ56GyzYD+UHNm+4xYJQqycYO4NvtrSXUAohq6i0CthxAfszhvFPJs/0HvjCYuoKCnutuYKPixrjYvrzRV3aTf0xWvkluMrOIL+JQVW+ZCOnxi+xaRRzESJezRyYQV/hldvSMV29gEzXkDXPAQLgEscYDO8MD+uUDruzAfLphzkZ22skiZWcQfjhGM93p4TJ1Ii4weBjyGZkAY/5g7Iz3dcFPU9QVnOyjG7nBV9EErT7EsWz0/bYqfqMK+42r7+olsFQZQtfAZBFfMhB6tCxiPMaWYnAexeEVfez3khTkoBe4+sKG6sY6R+5jxEjBwyfXr2mxGwa2jVNxZ65t2fJ/Fe1SczUld6YkmSdGMIUjR0T63MM3CrTRLme37M3WibTU9L5sXDm1CA9GL09ahqpdqDKEKmIYy4bxUfD0m1qGuoA5s1kiVab5XuDwmG2FDm2mFUk9nQgiinShHSUrd5QjmstqJ5DoGiL6k/B+TZeSTqFcskMbXOugi/j7TqHrVy6gJxVFxVTXH47qGdJVPA3vAeIuQ5oqPykeM8FcsNgpL1L8bSe8eKQ7iH4bh5kKL0Y7BUb66Waa52ZaPBwkBAyOvDMp3qs1XuCym8tpdB/eOFDWLXSzON7mxlYLYj+Dc6nGoGi+iItjbmwJKCS/tVjMV96hKHiHjPqEHDv/tscoOqrHyLFz8B1J6H5uzlrEqD/JqDspfmFoE6kJLA5M9hGHo4rTSr1SBz1mx3S7Yyh7ssfA5KjcSxIlgeoxaM1JEthBa0lOuo/krPs4hP0ew0keQ8XhYgxGNxuSPNq6xrzKZvwbcpuWOQjBaTb141EUN6O9ECc+Sg4BDKmYBwY1OuQNHi0zKkabWxHsrM9bcc7xQmfbyBSoeRxbY5VkSvIPdDwnNE7z4r5kcSDQKEUaMVxP8zJ0ew3PeCPludyPmlSMY7i1C01i821CYz2/spaHxJqp5KdDlHxfmCucmY1/z8t8Hnf6noxUWmU+qzLeuPsqnM7SsKe+uHeuVmyjnWPvIx9eMlOnAQyujO6g0qBbjdCNgy9b5UJzpYc4sFTZcWXnUOZsgsKoFjG6XRwchnQ9rX7CreboIbaBZYHlO0VST3T7gEclWXA6xs/ux2cpUwe8iCPDJj1iw1FTaDa+yJHdqb2/L37y3I3iNxzpmp1G+h1nFp3x/CzuMefuYKb7ylXk819TOot2Cc2p2SXuiDZqn1OadOf+EWeOpAkfnhRQk4+UUO3ILGHE3b6yC/TKrqrkDMuo5bVjTIJDuDs9mo2baBh/R001e5ZVfVZTPdhnGVUtE6dPuqofaqYGUdQpin5LDDXbpzHj2QuiqA7hpnob7nsn05rg20uCb+/jE//oRgNffv3p25sbStFVxN6SG2QXUVj75c9/+ennX7/94ac/fffDl/+1JvQiOs42lSQpjUzepZimHWOSWZQWVWvR9Y5PW+FBH3u+KrOngTGhtX2nVDsqYeB5tBVXvzKPr66QQlxPYylqbo6YNUaZSml7Y0ISztMP8l1XrN0Ej+roU9V2OPxLN/GNFQZqtUXCxqoynl8UqQjFMMPlLv9iGRfRIzfcLU5sdGMgDg3YuVVZ726kXI7DbBXqnmyroiMI69TYnNl29GVrNI3XVOYrDZIpi5UcbWC60mhBBgEEVWaQsNFZCD+GjezXk5NWedKNzyHogvEXgyv/EkirzAhhrIAjyUsgulEqeBA8KvK7ZeGJdl+oITy6v/783Y+//LA+snaiS+82XeXO4smUVBhaorlJKVhZ1+YWJogN1ixWizWktktGgOGP74IDPHBSRSPYCnYTBzPAFyVhSdgkVpIoSZUclByVnJSchWg1RvdgdA8mrNI9GN2D0fqM1qebxFpNrKXjsEpLx3p8sR5frPXFoT49Pi2UaMWJHl+ie0h0D0nYWPeQ6B4S3UOie0h0D6nWnGrNqdacas2p1pxqzalWmGqFUt+s92rWezXrvZr1Fs16i2a9RbPeItdM2MCywHJhWpPepdmEJbES3YnRnRjdid63We/brPdrjrVCvV+z3q9Z79esqNVpbTjzpaFVzGEsunY0dy1sSoCtkGn7WVty9F8/T7OIVPH5nwUXxbnq0IkAvMnSB1Be5F+//OnbP/31Tz8wosEiTRCt1WbCl41cNykDEl45hgxqEyFt1iEyS924gI5jkMCeogz6SPW4jYYbpFwHs5Lob6vv3qe5Mlxhk/Ha0i3vuhjnr8W0J/0sCWaC+W+oZveUp7bjTi47gy09E1NrUMjWqV+a6Umx+ncVqtGJCFLUDrGaYeMeMiLlM/Jbu89g9ZrRRyGHSMlByImkt6I+TTmOnENxj/r/SaW6kOWFLC9leSnL6cJmFKNHWdwIyN9vBFriQBgJjycFbPrTHcaCTA71RLdIcyHH35MQCdrZp52iCDM6HaoadqRq2JK++6Sqnazi2ua3dLY/UNaO9zolVjAXLAUrwYtgI3gVXATfRIybWcKOMA+5HA0+u44wN3gjz83o1JwaQ/lF3Bdfq29X0995fiFR+V7E+28E6bvXlOtWZdZb6Tio7Hdw62fbOYkWarDm/UKp3Xwp+pIFRXNL35gbU7dvTN1+Vvu+MZX7hii6oPm90PGc8+J6jAUPG5nuZo5Rd9HWPJugEEP3dZCHWfbua6/RvYgWd0yApjVDQ8qih7eqrBiv5rrlsEIDec2fNbk5YVKHM5Tw1rJnPkPZw79QjghxKWd2jVYdGk724lEEaajI7TARFEVtKnJ7zIQcBU+qV/NidzrY2Z09Kk+OO4XroFUN+ZSvJKlFfLrstgo3kh6qnuihpQAzvMIkMzuHDpksSjP6Jf6E4aLHjJgSkKMZP7jRgz3KmTasoCGNeIKgDQ0B9ifJZi/VzCe2QRIh9Fea3r/DHwkyj7wB7qxyEV2O4q0K8+eaTQ6IURJvfaDM8vwG78OqzMxP+7VAlM/1UkwZDvPaVJFiIigpeOwwgWQi3/xidzrOnCHJoxQxuj4nxlIwkeWJ/JdcvktBeYbVw9oxoPPKTMtrPy9w2F/7hUVGnkZrqYTsEV37nB4NFSeSkVzrbjn9qe/crrOiRAW9omex6OhlN6HVKKrQZwJj+lcJ6KD5/LHQERWOKP+vgs0lbn7XSMe1gWJR14iKtMooP8sh64xZDpMPBI0fGX9b+cezEXnjC8/pQc9F9041X2vRSHwle2ypFCzqx3ZkFpsnsZBaUddcFLHA3U9UXLCiomxVODnaCyTTE3mNt3LJaDwdIMcOaupBGHmgRzZIIAeNY9eYQdppFv1i5GZBtRg5fjc6SG/M0epv44H/RbhRJJ5G63pmJJW9MDp1zJAglnEor0JOrjnZzyAWpIXxhV6FhbNKEuCYMUcL9yiKMOO1e4ibkF3o31UcXuWinvSfRkmy8rK+echiMyrrmweSqdAvpvQIEr6a6SYXYSo4/dnU5DLrjfcXWsFsm/s2S5EF3Rg/YdlBfHs6URnzmWx2oEcur4P0rSrpqoLus3LUk8otLS59PVV0ydlopx8lulF5tFeJKulGc5jshGgRguMgI+RbOdqO8grPrriBmaYezVYdahWL1QQsWiRWaShYi6BVjlFXhoiYoAulvrBxaXD1tmqvMsuZikaFXCh6Xa5W/sVMkIo2eVKrzKrZqa2qY0VmPKMGag8ZCvZBvPsk3c97NhNzjHs9cn0hslRBgjWXZKcl6yWbqVY0Mh8aHYJL0+4kWDtuf6PdXHKwPer/mGqq5vdkVDO4tB2UewFVOpP+thZWf5130lcqUlqkRuz7KD9+rTUaH56TaI7RTmt0oRl1gejRAqfx4k3yVkmmJBcSB1IoKYUkWuokiN7cxgEQCRqrRNfohlqF+AocOQieFGVFIZWVgrWi5KfUTDcgiwNLAksDOzwphuWCjeAsDglKiX2lfJpT5uU2X3CRtjqnRvCwz0aJRcl0M0fb86Ro1AIQqVI7tnsVMrQRbyXO3YERFdJjvFEMfZSu55GEqc7OT0Z+11OShJAYb48jJ1iE3ntOcPr+hob5/a1pIY/giGta/uiNN+NlmGFmcY+iSm16lgWWB1YEVgrrAqmU1EJGrWHUYqNugyim3Pz08z8eOIUV/Jf5cGcQHgjVlO4cfTvMiBI97Ylscch0SaZLcuJRtz3qto60wjJFXSWFTrLDk/xnqHH+EIklSKkV9Q3TfrlhhqGlvVs++aSCwE5kDJwFMUpiIQ8pR2MHiFESK0kCaZWlSg7BuxDIS2DrMrNzQoCFRWFJqiRUelRyUnJWkoXKK2XZSl4CC8t0f5nuD0GiILrjTHec6Y4z3XGmO86jXZwZyKerG2BHWt2sW8y6xay7nMM2ustZdznrLmfd5avW86r1vDLk5GIFKFNeXDJCTigkKiO1aSRee/SPb5/ZujhkbhRIJCQTzHVBLmUmnpBnF8VWyGe8vv/zyz989+uX//H9t7/8+vP3v/wSVAi3htLWbNUFZfimKoP04NnW4iVFOL5ti4vkNPX1uGCwbMdB9ADxwk/omFqdOFBMm1WysXAyLm7NHgo2TKTEIAGmsyLTW23F+tb5qZk04E2MtRCajBD3nONNzIuYNrD4pN6d/ZxuqiJnZYyWD7THTQ8EJ61WGU1QoKIaIpmCbFp9xSTTF5qqV72s4b6zkITpo3U6MsgC/940Y4ePZxObaCsrlwqqVuXymm6n9mLHZ7VfwBta1RgTVIgs3UyjNei0WolMq4VI9QM+ynXOCP4cFuJ6nDTGPT7SsTreBWjSuDPK/c5A+TsD5e8MlL+z6B0OWsRO1+j/1PMiMe0DJ7JiI3OhY/LS8Vp6FK8tOywg2TYSXKeoeooEb8a8rDYaP1e7oG0Pwdglu1AixaNqOw2cCu1AdfkWLdLqBh6NBg6bq07WdBI8byZoktGvRgLXjMLsGoxfvK5NnHKSIFKzm+5op3HDATNDBFXyhsG+Mx4uL2xjBOOnxkXbi3gbyesedH7WGBTeVyUND6oLE6baYeRaf6cNRKfauU+EWUaq3Ucz6ug8OHnBYFUf/MolHFkWrUiylEiJ8tOVvgjKTNgczkrv+WmCbPWLy9TW10JDY6doM2ylVItoHfcaq2qnmkOTcWyshKjKsJP7cBgLJoLpLpT1g3mzw+i02Cu0QNR46hn6OmHqzHEW4ZG//2wz/4FlWTjB1LjUs0TxMojXjXmugriU90IBl/LeM9KXt/3B9LLxMQ13nS6HQasZBRCmbIRusI7Wa7prw/wvYrTDfC5mJ3aCvsIalprbSaJG27JHf49sDGz+cFKV+9PEJ+Y5HNSEQEnXUTAS7ZiECEgJlJS4RY1H5Gh93MYN3nUAfFPkAPjKOh2mggeGB5rt9N8dWoQFjSkn98b/Ese9XOqIc3Uw5pYRdRxKDgzXDUNXmWt7eFaPoCDEnWZFh3RagjA8DPbFG4KDb7nOL8FJqq/eVtZuU/onma2BSmoyAbTOrKAzH7gu0fjzT79+/+XHb//xpx/+4cuP/8TQJeahX2jOccipmevI6GTL5VYnmbMoM/nA9WoO7Mewy5fR2pDR2OAgJiSEdJt5y3jH3PZ3cYNSRjek3NKpuVDad+GMvQszSh1mxOM2PXdhdGDIr80jdVPG6os8ir9tmGgFztuafivOUUAtW34kXBMRS5CWBmthODYtVtNN7S7tNExk+0bf5y6cK8xGYEOkVyIkiwJZmQksDiwUCCvDEo0Vw+cORI8AAwWGkb0EFpaFORJMiDszSWDpyj7Vfb1WEFJHk6cc0m/y4xqrVva1RQYbMjg5WSz1fz3CMzWNuFQez0SEt/hYNuwNxCrJlORCdNtU8LCJg/N4RjjS9eNwuF0InKNZoMo4KARJfrP7hS+BJ2YfTGcTXZEKHvapexgyOtS9ZVJBJtvxJdawPIdlwE/dkMluLuzuwSd9vcwIwWs7xqV6xJC+XXit2sfEaR5KpDa7IQO8K0MNGAvmJlXwV/R1C3dbf0U8Rkg7guXCQb6z8dOuCoJWLMZxjQmGgGMmUmIFpbk9YquiYbvvUP8ngqnggdjS1s9Qa98lQoJRRTOxRyOYCKaCh322TqPdHrqWyfLAisDKwKrA6sAuZDTmN1JeCktJKSZlpAATd0bOVekx1oSdlfgb9nlmz98j82J6uoWmhvMqNEwR9MjEE7Of2yynfVsG4hXNpxiaJ6GBrtAEc81CK6nl3I6uxbOE/GuzrsyvVbaD2TZ4MgmhhkgvCsnO0qox0beaqYq3PuekxSAwCPK5R/wu+hBvzSjz8Dx6dqUx3YBrzZa//uX7b5tvm6pHQ0ZZfJHtFaXejXDu/zfCt9etbK3q1AYN2N2Mv1ZGYozDFe3O35OxDNpuqo/2JHumambPomVB1UuFmLwOkrsFt//65b//85dfXZfiL75r8dMPf/3lyy8yxRFNKmFmIzGtrFM0dNnELkeHKBfb9dHXMWgvX0ejUcEHLA3ssI9Vs0xYBwkbURLn2mRBJee8hqtJSfd9o3bLxPbCTpRY9sjlV2g62YljBY+nveYMVVgm6WLYOVMQ5AxOaKEdnD+a10ltR6K24hHxeXfaaTxKD+uwiXyr4KLykW1ma0Fa0CNew9LgxnW/WJpz/gWJFit4IbeRXi/ZU8jX4aP4rr97oFI7IP8/7wd2ETl9ioOZcwdQbsN1yphxli+9xNP96yONHCACP6dFFxiIURLvJg2QqakksL9kvpRHI4jlnQ09MHbqGHykGhocjJT5hA50WXAw5T68GSFH3yrYzxjbU0UbLYwwXT1DqUFiJclz4E90UHJUclJyFqLVcA4sT3SJ1qvVaq2HfVhRK1UO0YsSNdjpgQx6IIMeyGAUdVsTliRKtLQJhXRXseLLzjToiVYTh220vlgPIg7VyEUYdJNE60u0vkTrS7S+JGysx5doxYkeaKJ70E0O28QOh7peDuBNr92bXrs3vb9virLbN6nyTap6k6reUFXJ2eE9ZoL5JqoLHv+6xBtWN4UAjZhXtCMe0RMW5adLmS9FIDHJmyx4M6KdrxgL7jX1OZGXw/HGqtA9vdzeymjboo/jRtH8tZ8ji76LRlllS0ZbDVmsLCwKSxIlByVHJaedcMXC7gdIrOQoxCjqtqnggaFeDMcqOGdVsLtqpFdbhSislbwoM0piCdCKFF+UhCWqkhGFbRMlqZKjkpMGfClqfSbahYJ5ovUZrc9ofWgSQEJ9ZyFaKNaKY6041opjrS/W+mKtL9b6dJNEq0m0mkSr0cJaVs9RqzjTsE07d4c3q2UokNfZQIKDJ0f0RlsGlbbvC5XUbUnJjLrc6GMc9qoYBWPP2LvoriKMcZ2oUbHQ1P2aMSgIhMFiIgjePVrRVs9FDL2ompWY3aT3BV2ensArV8LDvJqxLyLRv0pJjC2jRNcwrYWDRNU+R4KDlzdHJhRIttUrZ0x4mM792Vod7NGqL25X+zNioaL9DOeThXccKIIIYC8r3Sw1K41Xmqw0XelxpaeVngNd6zLrzsxagTmsdK0rXve77nYttJZZi5w2BvB/u6Xbnn5LoTz6N9rA/yNLlRdyTJXEtPUmFlmJF6skU5L/xmzvvViPwM67OQ8lgExj1Wim8Gi2cx5WkMb2sWkLLi5ZvNUbbzFB69T1nLoNJNOQtY9VJKKvZkxPlKjiuFGBCaOFTFilYuTm9KRKEev+4rBEplsPO0gSG5geZaI1J1pzojUnWj7VqtPoWRM9SGCESd5VJV1SMT3Rbc52J5PR89XzqGdh9BqY6KBESusl0CsQy1kpSFWJHGIiNUopKXTcTkBfTxvfxCXO6olS6HYSWYSGYqAejWD8kePiKeDPcnLy7sbIKRCs4Y7ngZpWHimTQHeJA+x/5FDRoxGMBZOdNMJiOwGznTf98dpvpz7P2uiuSEmCjJOYLXkVsvkjcw7Z9dE+436bZd/uZw8Pit4bOW3z2wLZQeH6N2Wxk/RZxTpISWsg3yrv/NV04jwWN+SncvpcRy9RSNA/0g4VfzSpuIo3j7DirvLNz1NqaqBcoVNqFuZ5+vEi0XnIdZs6EN348hRv942uuKpstNbGKcg/EpKezC5Yb53enArTZSzHHsuxx3LssZxbIusTrm/lP5Wp/VzokRKjJN5Pk+4Ir9bETeWIpnjjt5K51LO7VHfnFKsILrS7KENHjjLteiQY76ZhL3bTsG+ksI/sigpNV3pY6XGlp5WeA41fNo617YTtMlN7fNrLamNKdolkRLTRq6Sgvkq+6VbY4KjSB0eVPjjmSooPYiG9yMFBpmQ/brUM3kYBs3r3Hq4rBulrTrv2XtmewmpkGSMd02PyldIBMRZMdOLWSGcth7L2o/apd3/8P/8XJIjn81mgAAA=")).decode())
print("Frozen module sizes:", {k:len(v) for k,v in MODULES.items()})

## 1. Download frozen public count matrices

In [ ]:
URLS={
"GSE134781":"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE134nnn/GSE134781/suppl/GSE134781_counts.txt.gz",
"GSE134780":"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE134nnn/GSE134780/suppl/GSE134780_counts.txt.gz",
}
def download(url,path):
    path=Path(path)
    if path.exists() and path.stat().st_size>1000: return path
    r=requests.get(url,timeout=120); r.raise_for_status()
    tmp=path.with_suffix(path.suffix+".part"); tmp.write_bytes(r.content); tmp.replace(path)
    return path

paths={k:download(u,PUBRES/f"{k}_counts.txt.gz") for k,u in URLS.items()}
for k,p in paths.items():
    print(k,p,p.stat().st_size,hashlib.sha256(p.read_bytes()).hexdigest())

## 2. Read matrices and identify samples

In [ ]:
def read_counts(path):
    x=pd.read_csv(path,sep="\t",compression="gzip")
    x=x.loc[:,~x.columns.astype(str).str.startswith("Unnamed")]
    x=x.rename(columns={x.columns[0]:"gene_id"})
    return x

c3=read_counts(paths["GSE134781"])
c10=read_counts(paths["GSE134780"])

S3={
"WT_1":["3mo_skM_WT_1","GSM3968615"],"WT_2":["3mo_skM_WT_2","GSM3968616"],"WT_3":["3mo_skM_WT_3","GSM3968617"],
"L1002P_1":["3mo_skM_hetL1002P_1","GSM3968618"],"L1002P_2":["3mo_skM_hetL1002P_2","GSM3968619"],"L1002P_3":["3mo_skM_hetL1002P_3","GSM3968620"],
"X753_1":["3mo_skM_hetX753_1","GSM3968621"],"X753_2":["3mo_skM_hetX753_2","GSM3968622"],"X753_3":["3mo_skM_hetX753_3","GSM3968623"],
}
S10={
"WT_1":["10mo_skM_WT_1","GSM3968592"],"WT_2":["10mo_skM_WT_2","GSM3968593"],"WT_3":["10mo_skM_WT_3","GSM3968594"],"WT_4":["10mo_skM_WT_4","GSM3968595"],
"HL1002P_1":["10mo_skM_HL1002P_1","GSM3968596"],"HL1002P_2":["10mo_skM_HL1002P_2","GSM3968597"],"HL1002P_3":["10mo_skM_HL1002P_3","GSM3968598"],"HL1002P_4":["10mo_skM_HL1002P_4","GSM3968599"],
"HH_1":["10mo_skM_HH_1","GSM3968600"],"HH_2":["10mo_skM_HH_2","GSM3968601"],"HH_3":["10mo_skM_HH_3","GSM3968602"],
}

def resolve_cols(df,spec):
    cols=list(map(str,df.columns))
    out={}
    for alias,candidates in spec.items():
        hits=[]
        for c in cols:
            if any((cand==c) or (cand.lower() in c.lower()) for cand in candidates):
                hits.append(c)
        if len(hits)!=1:
            raise RuntimeError(f"{alias}: expected one column, got {hits}. Columns={cols[:30]}")
        out[alias]=hits[0]
    return out

m3=resolve_cols(c3,S3); m10=resolve_cols(c10,S10)
print("3mo:",m3); print("10mo:",m10)

## 3. MGI one-to-one mouse→human orthology


In [ ]:
# FINAL PRIMARY ORTHOLOGY SOURCE:
# MGI HOM_ProteinCoding.rpt = "Mouse Protein Coding Genes having one-to-one Orthology with Human Genes".
# Stable public report; no BioMart dependency.
#
# IMPORTANT: "one-to-one" is a gene-relationship property, not necessarily a one-row-per-symbol
# serialization guarantee. We therefore validate MGI<->HGNC identity first, then collapse only
# source rows that encode the same gene relationship and differ solely in mouse NCBI Gene ID.

MGI_URL="https://www.informatics.jax.org/downloads/reports/HOM_ProteinCoding.rpt"
mgi_path=PUBRES/"HOM_ProteinCoding.rpt"

def download_text(url,path):
    path=Path(path)
    if path.exists() and path.stat().st_size>100_000:
        return path
    rr=requests.get(
        url,
        timeout=180,
        headers={"User-Agent":"MVA-NB02B/1.0.3 non-commercial research"}
    )
    rr.raise_for_status()
    if len(rr.content)<100_000:
        raise RuntimeError(
            f"MGI orthology response unexpectedly small: {len(rr.content)} bytes"
        )
    tmp=path.with_suffix(path.suffix+".part")
    tmp.write_bytes(rr.content)
    tmp.replace(path)
    return path

mgi_path=download_text(MGI_URL,mgi_path)
mgi_sha256=hashlib.sha256(mgi_path.read_bytes()).hexdigest()

mgi=pd.read_csv(
    mgi_path,
    sep="\t",
    header=None,
    names=[
        "mgi_id","mouse_symbol","mouse_ncbi_gene_id",
        "hgnc_id","human_symbol","human_ncbi_gene_id"
    ],
    dtype=str
).fillna("")

# Normalize whitespace only; never rewrite identifiers or symbols.
for c in mgi.columns:
    mgi[c]=mgi[c].astype(str).str.strip()

assert len(mgi)>=16000, f"Implausibly small MGI 1:1 table: {len(mgi)}"
for c in ["mgi_id","mouse_symbol","hgnc_id","human_symbol","human_ncbi_gene_id"]:
    assert mgi[c].ne("").all(), f"Blank required field in MGI report: {c}"

# Validate the biological 1:1 relation on stable mouse/human gene identities.
# Any true one-to-many / many-to-one conflict must fail hard.
checks={
    "mgi_id_to_hgnc_id": mgi.groupby("mgi_id")["hgnc_id"].nunique(),
    "hgnc_id_to_mgi_id": mgi.groupby("hgnc_id")["mgi_id"].nunique(),
    "mouse_symbol_to_mgi_id": mgi.groupby("mouse_symbol")["mgi_id"].nunique(),
    "human_symbol_to_hgnc_id": mgi.groupby("human_symbol")["hgnc_id"].nunique(),
}
bad={k:v[v!=1] for k,v in checks.items() if (v!=1).any()}
assert not bad, (
    "MGI report violates one-to-one gene identity after parsing: "
    + "; ".join(f"{k} n_bad={len(v)} examples={v.head().to_dict()}" for k,v in bad.items())
)

# Preserve source-level duplicates for audit. In the 2026-09-08 MGI report, Slc35f3/SLC35F3
# appears twice with the same MGI/HGNC/human identity but two mouse NCBI Gene IDs.
source_dupes=mgi[
    mgi.duplicated(["mgi_id","hgnc_id"],keep=False)
].sort_values(["mgi_id","hgnc_id","mouse_ncbi_gene_id"]).copy()

relation_cols=[
    "mgi_id","mouse_symbol","hgnc_id","human_symbol","human_ncbi_gene_id"
]
strict=mgi[relation_cols].drop_duplicates().reset_index(drop=True)

# After removing only external-ID serialization redundancy, every mouse and human symbol must be unique.
assert not strict.mgi_id.duplicated().any(), "MGI IDs must be unique after relationship normalization."
assert not strict.hgnc_id.duplicated().any(), "HGNC IDs must be unique after relationship normalization."
assert not strict.mouse_symbol.duplicated().any(), "Mouse symbols must be unique after relationship normalization."
assert not strict.human_symbol.duplicated().any(), "Human symbols must be unique after relationship normalization."
assert len(strict)==mgi.mgi_id.nunique()==mgi.hgnc_id.nunique(), (
    "Normalized pair count does not equal unique MGI/HGNC gene identities."
)

mgi_audit={
    "raw_rows":int(len(mgi)),
    "normalized_one_to_one_pairs":int(len(strict)),
    "source_rows_collapsed":int(len(mgi)-len(strict)),
    "duplicated_source_gene_relations":int(source_dupes[["mgi_id","hgnc_id"]].drop_duplicates().shape[0]),
    "raw_sha256":mgi_sha256,
}
if len(source_dupes):
    source_dupes.to_csv(
        PUBLIC/"NB02B_MGI_SOURCE_REDUNDANCY_AUDIT_v1_0_3.csv",
        index=False
    )
(PUBLIC/"NB02B_MGI_ORTHOLOGY_AUDIT_v1_0_3.json").write_text(
    json.dumps(mgi_audit,indent=2)
)

MAPPING_MODE="MGI_PROTEIN_CODING_ONE2ONE"
MAPPING_SOURCE="MGI HOM_ProteinCoding.rpt"

print("MGI raw rows:",len(mgi))
print("MGI normalized one-to-one pairs:",len(strict))
print("Source rows collapsed:",len(mgi)-len(strict))
print("SHA256:",mgi_sha256)
if len(source_dupes):
    print("Source serialization redundancy (audited, not a 1:1 conflict):")
    display(source_dupes)


## 4. Map count rows to human ortholog symbols

In [ ]:
# Count matrices use mouse gene symbols directly.
mouse_to_human=strict.set_index("mouse_symbol")["human_symbol"].to_dict()
mouse_to_human_upper={str(k).upper():v for k,v in mouse_to_human.items()}

def map_counts(df):
    z=df.copy()
    z["raw_id"]=z.gene_id.astype(str)
    z["id_clean"]=z.raw_id.str.replace(r"\.\d+$","",regex=True)

    # Exact symbol first; case-normalized only as a deterministic backup.
    z["human_symbol"]=[
        mouse_to_human.get(s,mouse_to_human_upper.get(str(s).upper()))
        for s in z.id_clean.astype(str)
    ]
    z=z[z.human_symbol.notna() & (z.human_symbol!="")].copy()

    numeric=[c for c in z.columns if c not in {"gene_id","raw_id","id_clean","human_symbol"}]
    for c in numeric:
        z[c]=pd.to_numeric(z[c],errors="coerce").fillna(0)

    # Count matrices can contain multiple rows resolving to the same ortholog symbol.
    z=z.groupby("human_symbol",as_index=False)[numeric].sum()
    print("Mapped genes:",len(z),"mode:",MAPPING_MODE)
    return z

h3=map_counts(c3)
h10=map_counts(c10)

coverage_rows=[]
for mod,members in MODULES.items():
    den=len(set(members))
    for dataset,mapped in [("GSE134781",h3),("GSE134780",h10)]:
        present=len(set(mapped.human_symbol).intersection(set(members)))
        coverage_rows.append({
            "dataset":dataset,
            "module":mod,
            "present":present,
            "frozen_total":den,
            "coverage":present/den if den else np.nan,
            "mapping_mode":MAPPING_MODE
        })

module_coverage=pd.DataFrame(coverage_rows)
display(module_coverage.sort_values(["module","dataset"]))
module_coverage.to_csv(PUBLIC/"NB02B_BUBR1_MUSCLE_MODULE_COVERAGE_v1_0_3.csv",index=False)

# Coverage is reported, not converted into a new post-hoc biological gate.
# The preregistration freezes modules, contrasts, |effect| threshold, LOO stability, and
# mitochondrial recurrence. It does not preregister a >=0.65 orthology-coverage cutoff.
# The scoring function below already refuses modules with <5 mapped genes.
primary_mito_coverage={}
for mod in ["Mito_translation","OXPHOS"]:
    z=module_coverage[module_coverage.module==mod].copy()
    assert len(z)==2, f"Missing coverage rows for {mod}"
    assert int(z.present.min())>=5, f"{mod} has too few mapped genes for module scoring."
    primary_mito_coverage[mod]={
        "min_present_genes":int(z.present.min()),
        "frozen_total":int(z.frozen_total.iloc[0]),
        "min_coverage_fraction":float(z.coverage.min()),
    }

LOW_COVERAGE_NOTICE_THRESHOLD=0.65
low_notice=module_coverage[module_coverage.coverage<LOW_COVERAGE_NOTICE_THRESHOLD].copy()
if len(low_notice):
    print(
        "QC NOTICE only: modules below descriptive 0.65 coverage threshold; "
        "this is NOT a preregistered call gate."
    )
    display(low_notice[["dataset","module","present","frozen_total","coverage"]])

print("Primary mitochondrial coverage QC:",json.dumps(primary_mito_coverage,indent=2))


## 5. Module scores and pre-specified contrasts

In [ ]:
def logcpm(mapped, sample_cols):
    mat=mapped.set_index("human_symbol")[sample_cols].astype(float)
    lib=mat.sum(axis=0)
    cpm=mat.div(lib,axis=1)*1e6
    return np.log2(cpm+0.5)

def module_scores(logmat):
    rows=[]
    genes=set(logmat.index)
    for mod, members in MODULES.items():
        use=sorted(genes.intersection(members))
        if len(use)<5:
            continue
        score=logmat.loc[use].mean(axis=0)
        for s,v in score.items():
            rows.append({"module":mod,"sample":s,"score":float(v),"n_genes":len(use)})
    return pd.DataFrame(rows)

cols3=list(m3.values()); cols10=list(m10.values())
sc3=module_scores(logcpm(h3,cols3)); sc10=module_scores(logcpm(h10,cols10))

def aliases_to_cols(mapping, prefix):
    return [v for k,v in mapping.items() if k.startswith(prefix)]
groups3={"WT":aliases_to_cols(m3,"WT_"),"L1002P":aliases_to_cols(m3,"L1002P_"),"X753":aliases_to_cols(m3,"X753_")}
groups10={"WT":aliases_to_cols(m10,"WT_"),"HL1002P":aliases_to_cols(m10,"HL1002P_"),"HH":aliases_to_cols(m10,"HH_")}

CONTRASTS=[
 ("3mo_L1002P_vs_WT",sc3,groups3["L1002P"],groups3["WT"]),
 ("3mo_X753_vs_WT",sc3,groups3["X753"],groups3["WT"]),
 ("3mo_X753_vs_L1002P",sc3,groups3["X753"],groups3["L1002P"]),
 ("10mo_HL1002P_vs_WT",sc10,groups10["HL1002P"],groups10["WT"]),
 ("10mo_HH_vs_WT",sc10,groups10["HH"],groups10["WT"]),
 ("10mo_HL1002P_vs_HH",sc10,groups10["HL1002P"],groups10["HH"]),
]

def exact_perm_p(a,b):
    vals=np.r_[a,b]; na=len(a)
    obs=abs(np.mean(a)-np.mean(b))
    diffs=[]
    for comb in itertools.combinations(range(len(vals)),na):
        ia=np.array(comb); mask=np.ones(len(vals),bool); mask[ia]=False
        diffs.append(abs(vals[ia].mean()-vals[mask].mean()))
    return (np.sum(np.asarray(diffs)>=obs-1e-12))/len(diffs)

def analyze(name,scores,A,B):
    out=[]
    for mod,g in scores.groupby("module"):
        d=g.set_index("sample").score
        if not set(A+B).issubset(d.index): continue
        a=d[A].to_numpy(); b=d[B].to_numpy()
        eff=float(a.mean()-b.mean())
        loo=[]
        for i in range(len(a)):
            for j in range(len(b)):
                loo.append(float(np.delete(a,i).mean()-np.delete(b,j).mean()))
        same=(all(x>0 for x in loo) if eff>0 else all(x<0 for x in loo) if eff<0 else False)
        robust=bool(abs(eff)>=0.15 and same)
        out.append({
            "contrast":name,"module":mod,"effect_log2":eff,
            "exact_permutation_p":exact_perm_p(a,b),
            "loo_min":min(loo),"loo_max":max(loo),
            "robust_directional":robust,
            "n_module_genes":int(g.n_genes.iloc[0]),
            "n_A":len(a),"n_B":len(b)
        })
    return out

rows=[]
for x in CONTRASTS: rows += analyze(*x)
res=pd.DataFrame(rows)
display(res.sort_values(["contrast","effect_log2"]))
res.to_csv(PUBLIC/"NB02B_BUBR1_MUSCLE_MODULE_EFFECTS_v1_0_3.csv",index=False)

## 6. Mitochondrial recurrence gate and allele-divergence summary

In [ ]:
BUBR1_VS_WT=["3mo_L1002P_vs_WT","3mo_X753_vs_WT","10mo_HL1002P_vs_WT","10mo_HH_vs_WT"]
mito={}
for mod in ["Mito_translation","OXPHOS"]:
    z=res[(res.module==mod)&(res.contrast.isin(BUBR1_VS_WT))].copy()
    pos=int(((z.robust_directional)&(z.effect_log2>0)).sum())
    neg=int(((z.robust_directional)&(z.effect_log2<0)).sum())
    spans_both_age = (
        ((z.contrast.str.startswith("3mo")) & z.robust_directional).any()
        and ((z.contrast.str.startswith("10mo")) & z.robust_directional).any()
    )
    if spans_both_age and max(pos,neg)>=2:
        call="RECURRENT_BUBR1_DOWNSTREAM"
        direction="UP" if pos>neg else "DOWN"
    else:
        call="CONTEXT_SPECIFIC"
        direction=None
    mito[mod]={"call":call,"direction":direction,"robust_up":pos,"robust_down":neg}

# Allelic divergence: X753 vs L1002P in young muscle and HL1002P vs HH in older muscle.
divergence=res[
    res.contrast.isin(["3mo_X753_vs_L1002P","10mo_HL1002P_vs_HH"])
    & res.robust_directional
].sort_values("effect_log2",key=lambda s:s.abs(),ascending=False)

print("Mito recurrence:",json.dumps(mito,indent=2))
print("Robust allele-divergent modules:")
display(divergence)

## 7. Sanitized public handoff

## 7. Descriptive concordance with v1.0.1 direct-symbol sensitivity run

In [ ]:
old_path=PUBLIC/"NB02B_BUBR1_MUSCLE_MODULE_EFFECTS.csv"
concordance=None
if old_path.exists():
    old=pd.read_csv(old_path)
    cmp=res.merge(
        old[["contrast","module","effect_log2","robust_directional"]],
        on=["contrast","module"],
        suffixes=("_MGI","_symbol")
    )
    if len(cmp):
        corr=float(cmp.effect_log2_MGI.corr(cmp.effect_log2_symbol,method="spearman"))
        sign_agree=float(
            (np.sign(cmp.effect_log2_MGI)==np.sign(cmp.effect_log2_symbol)).mean()
        )
        robust_agree=float(
            (cmp.robust_directional_MGI==cmp.robust_directional_symbol).mean()
        )
        concordance={
            "n_rows":int(len(cmp)),
            "spearman_effect":corr,
            "sign_agreement_fraction":sign_agree,
            "robust_call_agreement_fraction":robust_agree,
        }
        print(json.dumps(concordance,indent=2))
else:
    print("No v1.0.1 sensitivity output found; concordance skipped.")

In [ ]:
handoff={
 "schema":"MVA-NB02B-BUBR1-MUSCLE-1.0.3",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "status":"COMPLETE",
 "datasets":{
   "GSE134781":"3-month gastrocnemius; WT n=3, +/L1002P n=3, +/X753 n=3",
   "GSE134780":"8-10-month skeletal muscle; WT n=4, H/L1002P n=4, H/H n=3"
 },
 "critical_variant_mapping":"mouse L1002P models human BUBR1 L1012P; it is a nearby-domain comparator, not N1002K",
 "primary_modules":"same frozen 14 NB02 GO-derived modules",
 "mapping_mode":MAPPING_MODE,
 "mapping_source":MAPPING_SOURCE,
 "mapping_interpretation":"primary MGI protein-coding one-to-one orthology analysis",
 "mapping_sha256":mgi_sha256,
 "mapping_normalization_audit":mgi_audit,
 "primary_mito_coverage":primary_mito_coverage,
 "coverage_qc_note":"coverage is descriptive QC only; no >=0.65 gate was preregistered",
 "mitochondrial_recurrence":mito,
 "n_robust_allele_divergent_modules":int(len(divergence)),
 "robust_allele_divergent_modules":divergence[
     ["contrast","module","effect_log2","exact_permutation_p"]
 ].to_dict(orient="records"),
 "interpretation":{
   "if_mito_recurrent":"mitochondrial modules can be retained as recurrent BUBR1 downstream state, not causal rescue target",
   "if_mito_context_specific":"strengthens original NB02 conclusion that mitochondrial state is allele/context dependent",
   "allele_divergence":"supports mechanism-gated treatment logic rather than gene-only or BUBR1-level-only treatment logic"
 },
 "sensitivity_concordance":concordance,
 "drug_selection":False,
 "privacy":{"private_patient_data_used":False}
}
(PUBLIC/"NB02B_BUBR1_MUSCLE_HANDOFF_v1_0_3.json").write_text(json.dumps(handoff,indent=2))
print(json.dumps(handoff,indent=2))